### Infix to Postfix

Consider the following attribute grammar for converting infix expressions to postfix expressions:

    expression(p) → ws term(p) { '+' ws term(q) « p := p + ' ' + q + ' +' » }
    term(p) → factor(p) { '*' ws factor(q) « p := p + ' ' + q + ' *' » }
    factor(p) → (integer(p) | '(' expression(p) ')' ) ws
    integer(p) → digit(p) { digit(q) « p := q + p» } ws
    digit(p) → '0' « p := '0' » | ... | '9' « p := '9' »
    ws → { ' ' }

The implementation below contains several simplifications:
- the test `sym ∈ {'0', '1', … '9'}` is implemented by `'0' <= sym <= '9'`,
- if `sym` is a digit, it is converted to an integer by `ord(sym) - ord('0')`.

In [ ]:
pos: int; sym: str; src: str

def nxt():
    global pos, sym
    if pos < len(src): sym, pos = src[pos], pos + 1
    else: sym = chr(0) # end of input symbol

def expression() -> str:
    # expression(p) → term(p) { '+' ws term(q) « p := p + ' ' + q + ' +' » }
    ws(); p = term()
    while sym == '+': nxt(); ws(); p += ' ' + term() + ' +'
    return p

def term() -> str:
    # term(p) → factor(p) { '*' ws factor(q) « p := p + ' ' + q + ' *' » }
    p = factor()
    while sym == '*': nxt(); ws(); p += ' ' + factor() + ' *'
    return p

def factor() -> str:
    # factor(p) → (integer(p) | '(' expression(p) ')' ) ws
    if '0' <= sym <= '9': p = integer()
    elif sym == '(':
        nxt(); p = expression()
        if sym == ')': nxt(); ws()
        else: raise Exception("')' expected at " + str(pos))
    else: raise Exception("invalid character at " + str(pos))
    return p

def integer() -> str:
    # integer(p) → digit(p) { digit(q) « q := q + p» } ws
    p = digit()
    while '0' <= sym <= '9': p += digit()
    ws()
    return p

def digit() -> str:
    # digit(p) → '0' « p := '0' » | ... | '9' « p := '9' »
    # '0' <= sym <= '9'
    p = sym; nxt()
    return p

def ws():
    # ws → { ' ' }
    while sym == ' ': nxt()

def convert(s) -> str:
    global src, pos;
    src, pos = s, 0; nxt(); p = expression()
    if sym != chr(0): raise Exception("unexpected character at " + str(pos))
    return p

Extend the translator with unary and binary minus (`-`), integer division `/`, and exponentiation (`^`). First, extend the attribute grammar. Define operator precedence such that
- unary `-` binds tighter than all other binary operators, e.g., `-2^2` is converted to `2 ~ 2 ^`,
- binary `-` binds as tight as `+` and is left-associative, e.g. `3 - 2 - 1` is converted to `3 2 - 1 -`,
- `/` binds as right as `*` and is left-associative, e.g. `2 * 3 / 2 = 3` is converted to `2 3 * 2 /`,
- `^` binds tighter than all other binary operators and is right-associative, e.g. `2^2^3` is converted to `2 2 3 ^ ^`.

Note that the translation uses `~` for unary minus to distinguish it from `-` for binary minus.

YOUR ANSWER HERE

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

Here are some test cases:

In [ ]:
assert convert("-2^2") == "2 ~ 2 ^"
assert convert("3 - 2 - 1") == "3 2 - 1 -"
assert convert("2 * 3 / 2") == "2 3 * 2 /"
assert convert("2 ^ 2 ^ 3") == "2 2 3 ^ ^"